In [2]:
import joblib
import pandas as pd
from scipy.stats import spearmanr

from churnlens.explainability.shap_explainer import explain
from churnlens.features.preprocessing import FEATURES, TARGET
from churnlens.models.pipelines import build_pipeline

train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")
X_train, y_train = train[FEATURES], train[TARGET]
X_test = test[FEATURES]

explainer_pipe = build_pipeline("logistic_regression").fit(X_train, y_train)
production = joblib.load("../artifacts/churn_model_v1.joblib")
print("loaded model version:", production["version"])

loaded model version: v1


In [3]:
base_scores = explainer_pipe.decision_function(X_test)
prod_probs = production["model"].predict_proba(X_test)[:, 1]
print("rank correlation:", round(spearmanr(base_scores, prod_probs).correlation, 4))

rank correlation: 1.0


In [5]:
import numpy as np

grouped, baseline = explain(explainer_pipe, X_train, X_test)
print("shape:", grouped.shape, "baseline log-odds:", round(baseline, 3))

reconstructed = baseline + grouped.sum(axis=1).to_numpy()
print("additivity holds:", np.allclose(reconstructed, base_scores, atol=1e-6))

shape: (1409, 19) baseline log-odds: -0.671
additivity holds: True


In [6]:
importance = grouped.abs().mean().sort_values(ascending=False)
importance.round(3)

tenure              1.030
InternetService     0.588
MonthlyCharges      0.580
Contract            0.546
TotalCharges        0.399
StreamingMovies     0.234
StreamingTV         0.222
OnlineSecurity      0.192
PaymentMethod       0.171
TechSupport         0.166
PaperlessBilling    0.162
MultipleLines       0.162
OnlineBackup        0.111
DeviceProtection    0.107
Dependents          0.097
SeniorCitizen       0.041
gender              0.016
Partner             0.009
PhoneService        0.004
dtype: float64